# Data Collection

In [1]:
# Libraries below
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
from datetime import datetime

## Begin Data Collection

In [10]:
# Set up API key and Youtube API Client
API_KEY = "" # API key as string
youtube = build("youtube", "v3", developerKey = API_KEY)

In [18]:
# Input Channel ID
# Checked ID by searching channelId in "View Page Source" on YouTube video. 
CHANNEL_ID = "UC86BKGkw3Giit6yTAntInfQ"  # @Sinvicta Channel ID as string

In [3]:
# Set up date range for data collection
## For this project, a sample of the modern "Repentance Era" of the channel
## Range: March 31, 2021 ("Repentance #1") to June 30, 2025
START_DATE = datetime(2021, 3, 31) # Year, Month, Day
END_DATE = datetime(2025, 6, 30)

In [4]:
# Function to get channel statistics
def get_channel_stats(channel_id):
    try:
        request = youtube.channels().list(
            part="statistics,snippet",
            id=channel_id
        )
        response = request.execute()
        stats = response["items"][0]["statistics"]
        snippet = response["items"][0]["snippet"]
        return {
            "title": snippet["title"],
            "subscriber_count": stats.get("subscriberCount", "Hidden"),
            "video_count": stats["videoCount"],
            "view_count": stats["viewCount"],
            "published_at": snippet["publishedAt"]
        }
    except HttpError as e:
        print(f"Error fetching channel stats: {e}")
        return None

In [5]:
# Function to get video details within date range
def get_channel_videos(channel_id, start_date, end_date):
    try:
        # Get uploads playlist ID
        request = youtube.channels().list(
            part="contentDetails",
            id=channel_id
        )
        response = request.execute()
        uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

        # Fetch videos from uploads playlist
        videos = []
        next_page_token = None
        while True:
            request = youtube.playlistItems().list(
                part="snippet,contentDetails",
                playlistId=uploads_playlist_id,
                maxResults=50,  # Max 50 per request
                pageToken=next_page_token
            )
            response = request.execute()
            for item in response["items"]:
                published_at = datetime.strptime(item["snippet"]["publishedAt"], "%Y-%m-%dT%H:%M:%SZ")
                # Filter videos by date range
                if start_date <= published_at <= end_date:
                    video_id = item["contentDetails"]["videoId"]
                    video_title = item["snippet"]["title"]
                    # Fetch video statistics
                    video_request = youtube.videos().list(
                        part="statistics",
                        id=video_id
                    )
                    video_response = video_request.execute()
                    stats = video_response["items"][0]["statistics"]
                    videos.append({
                        "video_id": video_id,
                        "title": video_title,
                        "published_at": published_at.strftime("%Y-%m-%d"),
                        "view_count": int(stats.get("viewCount", 0)),
                        "like_count": int(stats.get("likeCount", 0)),
                        "comment_count": int(stats.get("commentCount", 0))
                    })
                elif published_at < start_date:
                    # Stop if videos are older than start date (uploads are sorted newest to oldest)
                    return videos
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
        return videos
    except HttpError as e:
        print(f"Error fetching video data: {e}")
        return []

In [23]:
# Main execution
def main():
    # Fetch channel stats
    channel_stats = get_channel_stats(CHANNEL_ID)
    if channel_stats:
        print("Channel Statistics:")
        print(channel_stats)

    # Fetch video data
    videos = get_channel_videos(CHANNEL_ID, START_DATE, END_DATE)
    if videos:
        df = pd.DataFrame(videos)
        print("\nVideo Data (first 5 rows):")
        print(df.head())
        # Save to CSV
        df.to_csv("sinvicta_videos_2021_2025.csv", index=False)
        print(f"\nSaved {len(videos)} videos to sinvicta_videos_2021_2025.csv")
    else:
        print("No videos found or error occurred.")

if __name__ == "__main__":
    main()

Channel Statistics:
{'title': 'Sinvicta', 'subscriber_count': '335000', 'video_count': '7879', 'view_count': '216274443', 'published_at': '2007-04-09T20:47:08Z'}

Video Data (first 5 rows):
      video_id                                              title  \
0  ki1XIBfNlbM                            Too much or not enough?   
1  EVBo6XUhZYU  BADgic Mush?! -  The Binding Of Isaac Repentan...   
2  7O48BCQ-b_U  WE GOT A STRONG RUN?! -  The Binding Of Isaac ...   
3  NM0-5HFmb5o  BIG MAMA BOOM -  The Binding Of Isaac Repentan...   
4  Kf02jQ5NpwA  IN WAFER WE TRUST -  The Binding Of Isaac Repe...   

  published_at  view_count  like_count  comment_count  \
0   2025-06-29       17706        1126             28   
1   2025-06-29       27840        2521            135   
2   2025-06-28       29009        2585             78   
3   2025-06-27       23868        2151             69   
4   2025-06-26       20867        2168            108   

                                                tags

In [22]:
# Modified function to get video details with tags, category ID, and duration
def get_channel_videos(channel_id, start_date, end_date):
    try:
        # Get uploads playlist ID
        request = youtube.channels().list(
            part="contentDetails",
            id=channel_id
        )
        response = request.execute()
        uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

        # Fetch videos from uploads playlist
        videos = []
        next_page_token = None
        while True:
            request = youtube.playlistItems().list(
                part="snippet,contentDetails",
                playlistId=uploads_playlist_id,
                maxResults=50,  # Max 50 per request
                pageToken=next_page_token
            )
            response = request.execute()
            for item in response["items"]:
                published_at = datetime.strptime(item["snippet"]["publishedAt"], "%Y-%m-%dT%H:%M:%SZ")
                # Filter videos by date range
                if start_date <= published_at <= end_date:
                    video_id = item["contentDetails"]["videoId"]
                    video_title = item["snippet"]["title"]
                    # Fetch video statistics, snippet, and contentDetails
                    video_request = youtube.videos().list(
                        part="statistics,snippet,contentDetails",
                        id=video_id
                    )
                    video_response = video_request.execute()
                    video_data = video_response["items"][0]
                    stats = video_data["statistics"]
                    snippet = video_data["snippet"]
                    content_details = video_data["contentDetails"]
                    videos.append({
                        "video_id": video_id,
                        "title": video_title,
                        "published_at": published_at.strftime("%Y-%m-%d"),
                        "view_count": int(stats.get("viewCount", 0)),
                        "like_count": int(stats.get("likeCount", 0)),
                        "comment_count": int(stats.get("commentCount", 0)),
                        "tags": ",".join(snippet.get("tags", [])),  # Join tags into a string
                        "category_id": snippet.get("categoryId", ""),
                        "duration": content_details.get("duration", "")  # ISO 8601 format
                    })
                elif published_at < start_date:
                    # Stop if videos are older than start date (uploads are sorted newest to oldest)
                    return videos
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
        return videos
    except HttpError as e:
        print(f"Error fetching video data: {e}")
        return []